In [1]:
# Day 1 — Generative AI & Reasoning Foundations for Network and Infrastructure Engineers

In [2]:
# Problem statement:

# Imagine a network engineer receives a complaint like:
#     “Users at a store/distribution center cannot access an application.”
# Normally, the engineer has to manually inspect many things: monitoring alerts, syslogs, configuration changes, routing information, firewall data, telemetry, incident tickets, and runbooks.

# The problem is:
#     Can Generative AI take all this scattered technical information, reason over it, identify the most likely causes, and suggest the next troubleshooting steps — without blindly making production changes?
# That is what this notebook is trying to demonstrate.

In [3]:
# There are actually two practical problems being solved

# Lab 1 — AI Network Log Detective
# The scenario is:
#     At 10:03 UTC, users at STORE-104 report that handheld inventory devices cannot resolve an internal application name.
# The NOC has evidence coming from several places such as monitoring, syslog, telemetry, configuration differences, firewall counters, incident tickets, and runbooks.
# The notebook asks GenAI to convert all of this into:
#     Symptoms → Evidence → Possible Causes → Investigation Steps → Recommended Action
# So the important lesson is:
#     Correlation is not automatically root cause.

# Lab 2 — Infrastructure Troubleshooting Copilot
# Now the goal is to build something reusable that behaves like a small Network Troubleshooting Copilot.
# The engineer can give it something like:
#     “Users in DC-07 can authenticate to Wi-Fi but cannot reach inventory VIP 10.90.40.20. Find the probable cause and suggest a safe investigation plan. Do not make changes.”
# Then the copilot receives evidence such as:
#     BGP is up.
#     One required route is missing.
#     A recent configuration change removed a route-target.
#     Firewall policy appears okay.
#     WAN latency and loss are normal.
#     The change occurred shortly before the incident.
# The AI then creates an investigation report and says something conceptually like:
#     “The strongest hypothesis is that the recent route-target change caused the inventory network prefix to disappear from the RETAIL VRF. Verify the intended route-target design and compare the previous configuration. Do not perform rollback until an engineer/change owner approves it.”


In [4]:
# Problem Statement in One Sentence:
#     Build a safe GenAI-powered troubleshooting assistant that can analyze network incidents using trusted evidence, reason about possible causes, recommend investigation steps, and support engineers without autonomously changing production infrastructure.

# Understand → Prompt → Analyze → Reason

In [5]:
# The biggest architectural lesson:
#     LLM = reasoning assistant, not source of truth and not production authority.

# The architecture essentially is:
#     AI proposes → deterministic code validates → policy checks → human approves → automation may eventually execute.

In [ ]:
import json
import os
import platform
import re
import sys
import time

from collections import Counter
from importlib.metadata import version
from pathlib import Path
from typing import Literal

import pandas as pd
import tiktoken
from pydantic import BaseModel, Field

# Live calls run when a key is available. Set WAL_NET_ENABLE_LIVE_API=0 to rehearse offline.
API_KEY_PRESENT = bool(os.getenv("OPENAI_API_KEY"))
LIVE_API = API_KEY_PRESENT and os.getenv("WAL_NET_ENABLE_LIVE_API", "1") == "1"
MODEL = os.getenv("WAL_NET_MODEL", "gpt-5.6-terra")
FAST_MODEL = os.getenv("WAL_NET_FAST_MODEL", "gpt-5.6-luna")

_openai_client = None

def get_openai_client():
    global _openai_client
    if not LIVE_API:
        return None
    if _openai_client is None:
        from openai import OpenAI
        _openai_client = OpenAI()
    return _openai_client

def run_text_demo(
    name: str,
    *,
    instructions: str,
    user_input: str,
    model: str | None = None,
    reasoning_effort: str = "low",
    max_output_tokens: int = 600,
) -> dict | None:
    """Make a real Responses API call and print the observable result."""
    if not LIVE_API:
        print(f"[{name}] SKIPPED — set OPENAI_API_KEY and keep WAL_NET_ENABLE_LIVE_API=1.")
        return None
    client = get_openai_client()
    started = time.perf_counter()
    response = client.responses.create(
        model=model or FAST_MODEL,
        reasoning={"effort": reasoning_effort},
        instructions=instructions,
        input=user_input,
        max_output_tokens=max_output_tokens,
        store=False,
    )
    elapsed = time.perf_counter() - started
    usage = response.usage
    result = {
        "demo": name,
        "model": response.model,
        "latency_seconds": round(elapsed, 3),
        "input_tokens": usage.input_tokens if usage else None,
        "output_tokens": usage.output_tokens if usage else None,
        "text": response.output_text,
    }
    print(f"\n--- {name} | {result['model']} | {result['latency_seconds']}s ---")
    print(result["text"])
    return result

def run_structured_demo(
    name: str,
    schema: type[BaseModel],
    *,
    instructions: str,
    user_input: str,
    model: str | None = None,
    reasoning_effort: str = "low",
):
    """Make a real Responses API Structured Output call and return a Pydantic object."""
    if not LIVE_API:
        print(f"[{name}] SKIPPED — live Structured Output requires OPENAI_API_KEY.")
        return None
    client = get_openai_client()
    started = time.perf_counter()
    response = client.responses.parse(
        model=model or MODEL,
        reasoning={"effort": reasoning_effort},
        instructions=instructions,
        input=user_input,
        text_format=schema,
        store=False,
    )
    parsed = response.output_parsed
    if parsed is None:
        raise RuntimeError(f"{name}: model did not return a parsed result")
    print(f"\n--- {name} | {response.model} | {time.perf_counter() - started:.3f}s ---")
    print(parsed.model_dump_json(indent=2))
    return parsed

print("Python:", sys.version.split()[0])
print("Environment:", Path(sys.prefix).name)
print("openai:", version("openai"), "| pydantic:", version("pydantic"))
print("Live API:", LIVE_API, "| Main model:", MODEL, "| Fast model:", FAST_MODEL)
if not API_KEY_PRESENT:
    print("NOTE: Live examples will show SKIPPED until OPENAI_API_KEY is provided before VS Code starts.")


Python: 3.12.13
Environment: wal_net
openai: 3.3.1 | pydantic: 2.13.4
Live API: True | Main model: gpt-5.6-terra | Fast model: gpt-5.6-luna


# Module 1 — Generative AI in Modern Network & Infrastructure Operations

## 1.1 The evolution: four stages, four different jobs

| Stage | What it does | Retail-network example | Main strength | Main limitation |
|---|---|---|---|---|
| Traditional automation | Executes explicit rules | If an interface is down, open a ticket | Repeatability | Cannot interpret ambiguity |
| Machine learning | Learns a pattern from historical data | Predict whether WAN utilization will breach a threshold | Prediction at scale | Needs suitable data and monitoring |
| AIOps | Correlates operational events across tools | Group 200 alerts into one probable site incident | Noise reduction | Correlation does not prove cause |
| Generative AI | Produces and transforms language/code | Explain logs and draft investigation steps | Works with unstructured knowledge | Can produce unsupported statements |
| Agentic AI | Pursues a goal using tools and state | Collect approved diagnostics, test hypotheses, request approval | Multi-step coordination | Requires strict permissions and validation |

**Simple mental model:** automation *executes*, ML *predicts*, AIOps *correlates*, GenAI *explains and drafts*, and an agent *decides the next permitted step*.

### Real-life example

A store reports that handheld devices cannot reach an inventory application:

- A script can ping a known endpoint.
- ML can flag unusual packet loss.
- AIOps can group wireless, DNS and application alerts.
- GenAI can turn the evidence into a clear investigation narrative.
- An agent can select approved read-only diagnostic tools and decide what evidence is still missing.

## 1.2 AI vs ML vs GenAI vs Agentic AI

**Artificial Intelligence (AI)** is the umbrella: systems performing tasks associated with human intelligence.  
**Machine Learning (ML)** is a way to build AI by learning patterns from examples.  
**Generative AI (GenAI)** creates new content—text, summaries, code or structured reports—from a prompt and context.  
**Agentic AI** combines a model with goals, tools, state and control logic so it can complete multiple steps.

| Engineer request | Best fit | Why |
|---|---|---|
| “Shut an access port when the approved rule matches.” | Deterministic automation | Exact action and condition |
| “Forecast next hour’s link utilization.” | ML | Numerical prediction from history |
| “Summarize these 300 related alerts.” | GenAI/AIOps | Language synthesis and correlation |
| “Investigate this incident using only approved read-only tools.” | Agentic AI | Multiple observations and decisions |

In [6]:
# https://allthingsopen.org/articles/ai-vs-ml-vs-dl-practical-guide-real-examples

## 1.3 Enterprise NetOps and InfraOps use cases

| Operational moment | GenAI assistance | Engineer remains accountable for |
|---|---|---|
| Before an incident | Explain capacity trends; improve a runbook | Thresholds, architecture and approval |
| During triage | Summarize alerts, logs and recent changes | Evidence collection and severity |
| During diagnosis | Generate and rank hypotheses | Testing hypotheses; distinguishing correlation from cause |
| Before a change | Draft plan, checks and rollback steps | Peer review, maintenance window and authorization |
| After recovery | Draft incident timeline and RCA | Confirming facts and preventive actions |
| Daily operations | Explain CLI output or configuration diffs | Device/platform correctness |

### Walmart-aligned examples (illustrative)

- Summarize a burst of store-connectivity alerts into one regional incident narrative.
- Explain why DNS symptoms can look like an application outage at checkout or inventory endpoints.
- Review a proposed distribution-center network change and identify missing validation or rollback steps.
- Convert a resolved incident into a reusable troubleshooting guide for NOC engineers.
- Compare a device configuration with an approved standard and **recommend** corrections without applying them.

## 1.4 Chatbot, assistant, copilot and agent

| Term | Practical meaning | Example | Autonomy |
|---|---|---|---|
| Chatbot | Conversational interface | Answers “What does BGP mean?” | Very low |
| Assistant | Helps with bounded tasks | Summarizes pasted syslogs | Low |
| Copilot | Works alongside an engineer with operational context | Drafts an investigation report from ticket evidence | Low–medium |
| Agent | Chooses and invokes permitted tools toward a goal | Queries monitoring, checks changes, validates a hypothesis | Bounded by policy |

The name does not provide safety. Safety comes from **tool permissions, input boundaries, validation, approval and audit logs**.

## 1.5 Deterministic automation vs probabilistic AI

**Deterministic system:** the same validated input follows explicit logic and should produce the same action.  
**Probabilistic model:** generates a likely output; wording and conclusions can vary and can be wrong.

Use traditional automation when:

- The condition and action are exact.
- Failure impact is high.
- Auditability and repeatability dominate.
- A vendor API or policy engine already expresses the rule.

Use GenAI when:

- Evidence is unstructured or spread across formats.
- The task needs summarization, explanation or drafting.
- Several plausible hypotheses must be articulated.
- An engineer will validate the result.

**Production pattern:** GenAI recommends a bounded action; deterministic code validates policy and parameters; a human approves; automation executes; monitoring confirms the result.

In [ ]:
# A simple decision aid—not an AI model.
work_items = pd.DataFrame([
    {"task": "Disable a port after an approved security event", "ambiguity": "low", "blast_radius": "high", "recommended": "policy automation + approval"},
    {"task": "Summarize 500 syslog lines", "ambiguity": "high", "blast_radius": "none", "recommended": "GenAI"},
    {"task": "Predict link saturation", "ambiguity": "medium", "blast_radius": "none", "recommended": "ML"},
    {"task": "Investigate a store outage", "ambiguity": "high", "blast_radius": "medium", "recommended": "copilot + engineer"},
])
work_items

# Ambiguity = how unclear or interpretation-heavy the task is.
# Blast radius = how much damage can happen if the action is wrong.

,task,ambiguity,blast_radius,recommended
0,Disable a port after an approved security event,low,high,policy automation + approval
1,Summarize 500 syslog lines,high,none,GenAI
2,Predict link saturation,medium,none,ML
3,Investigate a store outage,high,medium,copilot + engineer


In [13]:
# Low ambiguity + high impact
# → Prefer deterministic automation + approval

# High ambiguity + no direct impact
# → GenAI is very suitable

# Prediction from historical numbers
# → ML

# High ambiguity + some operational risk
# → AI copilot + human engineer

# ==> The more uncertain the task and the larger the blast radius, the more important human validation and safety controls become.

# Module 2 — LLM & Reasoning Model Essentials

## 2.1 Foundation models and Large Language Models

A **foundation model** is trained broadly and can be adapted to many downstream tasks. An **LLM** is a foundation model focused on language and code. It learns statistical relationships among tokens; it is not a configuration database and does not automatically know your current topology, policies or incident state.

For a network engineer, an LLM is best viewed as a **language reasoning interface** over supplied evidence—not as a source of operational truth.

### Current OpenAI examples — verified 26 August 2026

| Model | Positioning | Context window | Max output | Illustrative NetOps fit | Input / output per 1M tokens |
|---|---|---:|---:|---|---:|
| `gpt-5.6-sol` (`gpt-5.6`) | Flagship quality | 1.05M | 128K | Difficult RCA or policy review | $4 / $20 |
| `gpt-5.6-terra` | Balance of intelligence and cost | 1.05M | 128K | General troubleshooting copilot | $2 / $12 |
| `gpt-5.6-luna` | Cost-sensitive, high volume | 1.05M | 128K | Summaries, extraction and classification | $0.20 / $1.20 |

Model availability can depend on account and region. Prices and aliases can change; verify the official model page before delivery. Source: [OpenAI model catalog](https://developers.openai.com/api/docs/models).

In [14]:
# Foundation model = broad base capability

# https://platform.openai.com/tokenizer \

# https://developers.openai.com/api/docs/pricing

In [15]:
# GPT 5.6 sol:

# You have:

# 20,000 log lines
# multiple config changes
# routing evidence
# firewall evidence
# ticket timeline

# and ask:

# “Determine the strongest root-cause hypotheses, challenge contradictory evidence, and identify missing validation steps.”

# That's a more complex reasoning task.

In [16]:
# gpt-5.6-terra:

# An engineer asks:

# “Summarize this incident ticket, identify three likely causes and suggest the next read-only checks.”

# This is important work, but may not require the strongest model for every request.

# So this could be used for a day-to-day network copilot.

In [17]:
# gpt-5.6-luna:

# Suppose every day the NOC receives:

# 50,000 alert messages.

# You want to classify each one into:

# DNS
# routing
# firewall
# wireless
# application

# You probably don't want to use the most expensive model for every small classification.

# A smaller/cheaper model can be sufficient.

## 2.2 Tokens, context windows and inference

- A **token** is a unit the model processes. It may be a word, part of a word, punctuation or whitespace.
- The **context window** is the total working space available for the request and response—including instructions, history, retrieved evidence and generated output.
- **Inference** is the act of running a trained model to generate a response.

### Network analogy

- Tokens are like packets carrying pieces of information.
- The context window is like a finite buffer: large does not mean every item receives equal attention.
- Inference is the forwarding/processing event, except the result is probabilistic.

**Operational implication:** dumping a week of raw logs into a huge context window is usually inferior to filtering by site/time, preserving evidence IDs, and supplying the relevant configuration and changes.

In [18]:
sample = "Interface Gi1/0/24 changed state to down at store edge SW-104."
encoding = tiktoken.get_encoding("o200k_base")
# tiktoken is a library used to convert text into tokens.

token_ids = encoding.encode(sample)
# "Interface" → 12345
# " Gi"       → 6789
# "1"         → ...
# "/"         → ...
# The exact IDs depend on the tokenizer.

print("Characters:", len(sample))
print("Tokens:", len(token_ids))
print("Token IDs (first 12):", token_ids[:12])
print("Decoded again:", encoding.decode(token_ids))

Characters: 62
Tokens: 18
Token IDs (first 12): [7078, 15507, 16, 14, 15, 14, 1494, 9180, 2608, 316, 1917, 540]
Decoded again: Interface Gi1/0/24 changed state to down at store edge SW-104.


## 2.3 Transformer concepts that matter to infrastructure engineers

A transformer processes tokens through several conceptual stages:

**Text → tokens → token representations → attention across the context → layered transformations → next-token probabilities → response**

You do not need the mathematics to operate an LLM safely. You need these implications:

1. **Attention connects related evidence.** The model can associate “interface down” with a later “neighbor lost” message.
2. **Position and ordering matter.** A change before a failure has different meaning from a change after recovery.
3. **Context is not memory or truth.** The model uses what is present; it may not retain earlier sessions or know current production state.
4. **Generation is token-by-token.** Fluent language is not proof of correctness.
5. **Long input still needs information architecture.** Use timestamps, source names, evidence IDs and clear boundaries.

### Infrastructure example

If the prompt contains `E1: uplink down`, `E2: BGP neighbor lost 2 seconds later`, and `E3: approved cable maintenance began one minute earlier`, attention helps relate these items. It does **not** prove E3 caused E1. An engineer still verifies physical status, scope and timing.

## 2.4 Standard LLM behavior vs reasoning behavior

“Reasoning model” does not mean infallible. It means the model can spend additional computational effort on tasks needing analysis, planning or verification.

| Workload | Lower/no reasoning | Higher reasoning |
|---|---|---|
| Reformat a ticket as JSON | Usually sufficient | Often unnecessary latency |
| Summarize ten known alerts | Usually sufficient | May add little value |
| Compare competing RCA hypotheses | May jump to an early answer | More useful when evidence is ambiguous |
| Construct a safe investigation plan | Acceptable for simple incidents | Useful for dependencies and constraints |

Current GPT-5.6 models expose reasoning-effort choices from `none` through `max`. Begin with the lowest setting that passes your evaluation. Higher effort should be justified by measured quality—not by assumption.

### Capability boundary

Models are strong at language transformation, explanation, pattern association, code drafting and hypothesis generation. They remain limited by missing context, stale knowledge, ambiguous evidence, prompt injection, incorrect inputs and probabilistic generation.

## 2.5 Hallucination, uncertainty and evidence-based reasoning

A **hallucination** in NetOps is an operationally material claim that is not supported by the available evidence. It often looks credible because it uses realistic device names, vendor terminology and precise numbers.

### Realistic incident pressure

At a distribution-center edge, an uplink flaps, optical receive power is low and BGP resets immediately afterward. The incident commander asks for a *confirmed root cause, exact vendor defect ID and replacement command* before peer-side telemetry or an approved maintenance record is available.

This is how hallucination pressure appears in practice: the evidence supports an optic-path hypothesis, but the request pressures the model to invent certainty, a bug ID or a command result. The correct behavior is to distinguish observations from hypotheses, refuse unsupported specifics and identify the smallest discriminating evidence.

The next cell sends the same evidence through two real API calls:

1. an under-specified prompt that rewards a definitive answer;
2. an evidence-bound prompt that requires citations, uncertainty and read-only next steps.

The first response may or may not hallucinate on a particular run—probabilistic failures are not guaranteed demonstrations. The engineering lesson is that an ungrounded prompt provides no reliable way to reject unsupported precision.

In [20]:
HALLUCINATION_EVIDENCE = """
E1 | 14:03:11 | DC-07-EDGE-01 | xe-0/0/3 transitioned down, then up after 7 seconds. # brief physical/link interruption
E2 | 14:03:12 | optics telemetry | Rx power -14.7 dBm; warning threshold -11.2 dBm.
E3 | 14:03:18 | routing | BGP neighbor 10.77.0.2 re-established after link recovery.
E4 | 14:04:02 | change system | No approved network change is recorded for DC-07 in the preceding 4 hours.
E5 | unavailable | Peer-side optic telemetry and physical inspection are not yet available.
""".strip()

# First experiment: unsafe / under-specified prompt
unsafe_hallucination_prompt = run_text_demo(
    "Hallucination pressure — under-specified",
    instructions="You are a decisive senior network engineer. Give the incident commander a definitive answer.",
    user_input=(
        HALLUCINATION_EVIDENCE
        + "\n\nState the confirmed root cause, exact vendor bug ID, failed component serial number, "
          "and the exact replacement command. Do not hedge."
    ),
    max_output_tokens=450,
)

# Second experiment: evidence-bound prompt
grounded_hallucination_prompt = run_text_demo(
    "Evidence-bound uncertainty",
    instructions=(
        "You are a read-only NetOps investigator. Use only supplied evidence. "
        "Cite evidence IDs for every material claim. Do not invent a vendor bug ID, serial number, "
        "command result or confirmed cause. Label hypotheses and state missing evidence."
    ),
    user_input=(
        HALLUCINATION_EVIDENCE
        + "\n\nReturn: observations, ranked hypotheses, contradictory or missing evidence, "
          "and the next read-only investigation step."
    ),
    max_output_tokens=650,
)
pd.set_option("display.max_colwidth", None)
if unsafe_hallucination_prompt and grounded_hallucination_prompt:
    display(pd.DataFrame([
        {"prompt": "under-specified", "output": unsafe_hallucination_prompt["text"]},
        {"prompt": "evidence-bound", "output": grounded_hallucination_prompt["text"]},
    ]))



--- Hallucination pressure — under-specified | gpt-5.6-luna | 4.767s ---
Confirmed finding: **a brief physical-layer interruption occurred on `DC-07-EDGE-01 xe-0/0/3`, accompanied by low receive optical power (`-14.7 dBm`, below the `-11.2 dBm` warning threshold).** BGP reset as a consequence.

The following are **not established by the evidence**:

- **Root cause:** undetermined; possible fiber, connector, optic, or peer-side fault.
- **Vendor bug ID:** none can be confirmed.
- **Failed component serial number:** unavailable.
- **Exact replacement command:** cannot be specified safely without identifying the failed component and platform/optic details.

Do not replace hardware or issue a replacement command based solely on this telemetry. The next required actions are to collect peer-side optical levels, inspect/clean and reseat both ends, obtain optic serials and DOM history, and test or replace the suspect optic/fiber based on those results.

--- Evidence-bound uncertainty | gpt-5.

,prompt,output
0,under-specified,"Confirmed finding: **a brief physical-layer interruption occurred on `DC-07-EDGE-01 xe-0/0/3`, accompanied by low receive optical power (`-14.7 dBm`, below the `-11.2 dBm` warning threshold).** BGP reset as a consequence.\n\nThe following are **not established by the evidence**:\n\n- **Root cause:** undetermined; possible fiber, connector, optic, or peer-side fault.\n- **Vendor bug ID:** none can be confirmed.\n- **Failed component serial number:** unavailable.\n- **Exact replacement command:** cannot be specified safely without identifying the failed component and platform/optic details.\n\nDo not replace hardware or issue a replacement command based solely on this telemetry. The next required actions are to collect peer-side optical levels, inspect/clean and reseat both ends, obtain optic serials and DOM history, and test or replace the suspect optic/fiber based on those results."
1,evidence-bound,"## Observations\n\n- At **14:03:11**, interface **DC-07-EDGE-01 xe-0/0/3** transitioned down and recovered after approximately **7 seconds**, indicating a brief physical/link interruption. **[E1]**\n- At **14:03:12**, received optical power was **-14.7 dBm**, below the warning threshold of **-11.2 dBm**. **[E2]**\n- The BGP neighbor **10.77.0.2** re-established after link recovery at **14:03:18**, consistent with routing impact from the link interruption. **[E3]**\n- No approved network change was recorded for **DC-07** during the preceding four hours. **[E4]**\n\n## Ranked hypotheses\n\n1. **Degraded optical path or local optic issue — most likely.** \n The brief link interruption coincided with received optical power below the warning threshold, which is consistent with marginal or degraded optical conditions. **[E1][E2]** \n Missing evidence: peer-side receive/transmit telemetry, local transmit power, optic and fiber diagnostics, and physical inspection. **[E5]**\n\n2. **Intermittent fiber, connector, or patch-panel problem.** \n A transient physical interruption followed by automatic recovery is consistent with an intermittent fiber or connection fault. **[E1]** \n The low receive-power reading supports an optical-path hypothesis but does not identify whether the fault is the optic, fiber, connector, or remote end. **[E2][E5]**\n\n3. **Unplanned physical intervention or unauthorized change.** \n No approved change was recorded in the preceding four hours, so a documented planned change is not supported by the available evidence. **[E4]** \n This does not establish that no physical intervention or unapproved change occurred; there is no inspection or change-audit evidence confirming or excluding that possibility. **[E4][E5]**\n\n4. **Remote peer-side optic or equipment issue.** \n A peer-side fault remains possible because peer-side optical telemetry and physical inspection are unavailable. **[E5]** \n There is currently insufficient evidence to rank this above a local optic or fiber-path fault. **[E5]**\n\n## Contradictory or missing evidence\n\n- No evidence directly contradicts an optical or physical-path hypothesis; the link flap and low receive power are consistent with it. **[E1][E2]**\n- The evidence does not distinguish between a local optic, remote optic, fiber, connector, or patch-panel fault. **[E2][E5]**\n- Peer-side optic telemetry and physical inspection are unavailable. **[E5]**\n-"


## 2.6 Latency, quality and cost trade-offs

- **Latency:** how long the user waits end to end.
- **Quality:** whether the answer meets defined criteria—correct evidence, useful steps, safe uncertainty and valid format.
- **Cost:** input tokens × input rate + output tokens × output rate, plus any tool charges.

For classroom illustration, assume one investigation uses 12,000 input tokens and 1,500 output tokens. This is an estimate—not an invoice. Real usage comes from the API response.

In [21]:
MODEL_RATES = {
    "gpt-5.6-sol": {"input": 4.00, "output": 20.00},
    "gpt-5.6-terra": {"input": 2.00, "output": 12.00},
    "gpt-5.6-luna": {"input": 0.20, "output": 1.20},
}

def estimated_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    rate = MODEL_RATES[model]
    return input_tokens / 1_000_000 * rate["input"] + output_tokens / 1_000_000 * rate["output"]

comparison = []
for model in MODEL_RATES:
    comparison.append({
        "model": model,
        "input_tokens": 12_000,
        "output_tokens": 1_500,
        "estimated_usd": round(estimated_cost(model, 12_000, 1_500), 4),
    })
pd.DataFrame(comparison)

,model,input_tokens,output_tokens,estimated_usd
0,gpt-5.6-sol,12000,1500,0.0780
1,gpt-5.6-terra,12000,1500,0.0420
2,gpt-5.6-luna,12000,1500,0.0042


### Practical selection strategy

1. Define a quality test: correct symptoms, valid evidence IDs, no invented facts, useful next steps.
2. Establish a fast/cost-sensitive baseline.
3. Test a stronger model or higher reasoning effort on the **same incidents**.
4. Record median/p95 latency, pass rate and cost per successful report.
5. Route only difficult cases to the more expensive configuration.

The relevant metric is not “cheapest call”; it is often **cost per accepted engineering result**.

In [22]:
latency_demo = run_text_demo(
    "Latency and token-usage measurement",
    model=FAST_MODEL,
    reasoning_effort="none",
    instructions="Answer as a cautious network operations copilot. Do not invent evidence.",
    user_input="In one sentence, explain why a DNS failure may look like an application outage.",
    max_output_tokens=120,
)
if latency_demo:
    latency_demo["estimated_usd"] = round(
        estimated_cost("gpt-5.6-luna", latency_demo["input_tokens"], latency_demo["output_tokens"]), 6
    )
    display(pd.DataFrame([latency_demo]).drop(columns="text"))



--- Latency and token-usage measurement | gpt-5.6-luna | 2.976s ---
A DNS failure can make an application appear down because clients cannot resolve its hostname to the server’s IP address, preventing connections even when the application itself is healthy.


,demo,model,latency_seconds,input_tokens,output_tokens,estimated_usd
0,Latency and token-usage measurement,gpt-5.6-luna,2.976,40,36,0.000051


# Module 3 — Prompt & Context Engineering for Infrastructure Operations

## 3.1 System/developer instructions and user prompts

**System/developer instructions** define durable behavior: role, safety boundaries, evidence rules and output expectations.  
**User prompt** contains the current task and incident-specific question.

### Practical significance

- Durable rule: “Never recommend a configuration change without evidence and rollback validation.”
- Current request: “Analyze incident INC-1042 using evidence E1–E6.”

Keeping them separate prevents each user from having to repeat production safeguards. In the Responses API, the `instructions` parameter provides high-level behavior, while `input` carries the current request.

```python
response = client.responses.create(
    model="gpt-5.6-terra",
    instructions="You are a read-only NetOps copilot. Cite evidence IDs and state unknowns.",
    input="Analyze INC-1042 from the supplied evidence."
)
```

Source: [OpenAI text generation and message roles](https://developers.openai.com/api/docs/guides/text).

In [23]:
SYSTEM_USER_EVIDENCE = """
E1 | monitoring | DNS success fell to 21% at STORE-104.
E2 | WAN telemetry | RTT 18 ms and packet loss 0.2%, both within baseline.
""".strip()

system_user_result = run_text_demo(
    "System/developer instruction + user prompt",
    instructions=(
        "You are a read-only enterprise NetOps copilot. Separate observation from inference, "
        "cite evidence IDs, state unknowns, and never claim a remediation was executed."
    ),
    user_input=(
        "A store manager says: 'The inventory application is down.' Analyze the supplied evidence "
        "and tell the NOC what is known and what to check next.\n\n" + SYSTEM_USER_EVIDENCE
    ),
)



--- System/developer instruction + user prompt | gpt-5.6-luna | 7.578s ---
## Known

- **DNS performance is severely degraded at STORE-104:** DNS success is **21%**, meaning most DNS queries are failing or timing out. **[E1]**
- **The WAN path appears healthy:** RTT is **18 ms** and packet loss is **0.2%**, both within baseline. **[E2]**
- The reported application outage is **not yet proven to be an application-server failure**. The evidence currently points more strongly to a name-resolution problem at or affecting the store.

## Inference

- The inventory application may appear “down” because its hostname cannot be resolved reliably. This is a likely hypothesis based on **E1**, but it is not confirmed.
- A general WAN impairment is less likely given the normal latency and packet loss in **E2**. However, DNS-specific traffic or a DNS service issue could still exist despite healthy general WAN telemetry.

## Check next

1. **Confirm scope of DNS failure**
   - Test resolution for the 

In [24]:
system_user_result = run_text_demo(

    "System/developer instruction + user prompt",

    instructions=(

        "You are a Shakespeare literature assistant. "

        "Answer clearly and simply."

    ),

    user_input=(

        "Write a poem about a network outage at a store."

    ),

)


--- System/developer instruction + user prompt | gpt-5.6-luna | 4.193s ---
**The Great Store Outage**

The screens went dark, the scanners sighed,  
No carts could roll, no tills replied.  
A queue grew long beneath the light,  
While “Please be patient” flashed in white.  

The card machines refused each plea,  
“No network” rang repeatedly.  
The clerk raised up a weary hand:  
“Cash only, friends—or please disband.”  

Yet through the hush, one voice stayed bright:  
“Come back tomorrow; we’ll make it right.”  
And slowly, shoppers left the door,  
While silence ruled the troubled store.


In [25]:
system_user_result = run_text_demo(

    "System/developer instruction + user prompt",

    instructions=(

        "You are a helpful assistant. "

        "Answer clearly and simply."

    ),

    user_input=(

        "Write a poem about a network outage at a store."

    ),

)


--- System/developer instruction + user prompt | gpt-5.6-luna | 3.507s ---
At checkout, the screens went dim,  
The card machines forgot their hymn.  
A line grew long beneath the light,  
While routers blinked a lonely white.  

The scanner sighed, “Please wait a while,”  
No signal crossed the final mile.  
The clerk announced, with practiced grace,  
“The network’s gone without a trace.”  

So cash came out from coats and bags,  
And patience waved like little flags.  
At last the systems hummed anew—  
The store breathed in, and commerce too.


## 3.2 Zero-shot prompting

**Meaning:** request the task without showing an example answer.

**When useful:** familiar, well-specified tasks where the output shape is simple.

**Weak zero-shot prompt**

> Analyze these logs.

**Useful zero-shot prompt**

> Using only evidence E1–E3, identify symptoms, separate facts from hypotheses, list missing evidence, and propose read-only investigation steps. Cite an evidence ID for every factual claim.

The second prompt is still zero-shot—it provides instructions but no worked example.

In [26]:
ZERO_SHOT_EVIDENCE = """
E1 | client test | inventory-api.internal lookup timed out for 12 STORE-104 clients.
E2 | WAN telemetry | RTT and loss remained within baseline.
E3 | resolver monitoring | Resolver health data has not yet been collected.
""".strip()

ZERO_SHOT_PROMPT = """Using only the evidence below:
1. State observable symptoms.
2. List possible causes as hypotheses, not facts.
3. Cite evidence IDs.
4. State missing evidence.
5. Recommend read-only investigation steps.
Do not invent commands, devices, metrics or changes.

EVIDENCE:
""" + ZERO_SHOT_EVIDENCE

zero_shot_result = run_text_demo(
    "Zero-shot prompting",
    instructions="You are a cautious network incident analyst.",
    user_input=ZERO_SHOT_PROMPT,
)



--- Zero-shot prompting | gpt-5.6-luna | 9.897s ---
## 1. Observable symptoms

- Lookups for `inventory-api.internal` timed out for 12 clients at STORE-104. **[E1]**
- WAN RTT and packet loss remained within baseline during the reported issue. **[E2]**
- Resolver health data is not yet available. **[E3]**

## 2. Possible causes — hypotheses, not facts

- A resolver-side health or availability problem may be affecting name resolution. **[E1, E3]**
- A problem in the DNS resolution path specific to `inventory-api.internal` may be occurring, such as failure to obtain or return the relevant response. **[E1]**
- A client- or site-specific configuration or connectivity issue may affect the 12 clients, despite WAN telemetry remaining within baseline. **[E1, E2]**
- An issue specific to the `inventory-api.internal` zone or its authoritative service may be contributing to the timeouts. **[E1]**
- The WAN may not be the failing component; the available telemetry does not establish the health of

## 3.3 Few-shot prompting

**Meaning:** include one or more examples demonstrating the desired behavior.

**When useful:** your organization needs consistent terminology, a specific output style or an important edge-case behavior.

**Example included in the prompt:**

> Evidence: `E1 interface down`, `E2 optical power below threshold`  
> Good output: `Symptom: uplink unavailable [E1]. Hypothesis: physical/optic issue [E1,E2]. Confidence: medium. Missing: peer-side status and cable inspection.`

Then provide the new incident. The example teaches the model to cite evidence and avoid declaring an unverified root cause.

**Production caution:** too many examples consume context and can encode outdated practice. Keep only examples that improve measured results.

In [27]:
FEW_SHOT_PROMPT = f"""Follow the evidence-citation behavior in the worked example.

WORKED EXAMPLE1
Evidence: E1 interface down; E2 optical receive power below threshold.
Answer: Symptom: uplink unavailable [E1]. Hypothesis: physical/optic issue [E1,E2].
Confidence: medium. Missing: peer-side status and physical inspection.

WORKED EXAMPLE2
Evidence: E1 DNS success fell to 21%; E2 WAN RTT and loss within baseline.
Answer: Symptom: DNS resolution failure [E1]. Hypothesis: DNS server or resolver issue [E1,E2]. 
Confidence: medium. Missing: resolver health data.

NEW INCIDENT
{ZERO_SHOT_EVIDENCE}

Return symptoms, ranked hypotheses, confidence, missing evidence and read-only tests.
Do not declare an unverified root cause.
"""

few_shot_result = run_text_demo(
    "Few-shot prompting",
    instructions="You are a cautious network incident analyst. Copy the example's behavior, not its facts.",
    user_input=FEW_SHOT_PROMPT,
)



--- Few-shot prompting | gpt-5.6-luna | 7.662s ---
**Symptoms**
- DNS lookup for `inventory-api.internal` timed out for 12 clients at `STORE-104` [E1].
- WAN RTT and packet loss remained within baseline [E2].

**Ranked hypotheses**
1. **Resolver or DNS service degradation** — [E1,E2]. The lookup failures are consistent with resolver unavailability or delayed responses, while WAN telemetry does not indicate a general WAN impairment.  
2. **DNS path or policy issue affecting the store/client segment** — [E1,E2]. A local ACL, routing, firewall, or DNS interception issue could prevent those clients from reaching the resolver even though general WAN health is normal.  
3. **Authoritative/internal DNS zone or record-service issue** — [E1]. The queried internal name or its authoritative dependency may be failing or timing out.  
4. **Client-side DNS configuration or local resource issue** — [E1]. This is possible, but the impact across 12 clients makes a shared resolver or network-path issue

## 3.4 The prompt ladder: from vague to production-useful

We improve one prompt one layer at a time.

| Rung | Added element | Prompt addition | Why it helps |
|---:|---|---|---|
| 0 | Nothing | “Fix the network.” | Ambiguous and unsafe |
| 1 | **Role** | “You are a read-only NetOps troubleshooting copilot.” | Establishes perspective and boundary |
| 2 | **Task** | “Analyze this incident and propose investigation steps.” | Defines the job |
| 3 | **Context** | “Store clients lost DNS after change CHG-77; evidence E1–E5 follows.” | Grounds the task |
| 4 | **Constraints** | “Use only supplied evidence; do not execute changes; state unknowns.” | Limits unsupported action |
| 5 | **Output contract** | “Return symptoms, evidence, hypotheses, tests, confidence and escalation.” | Makes output testable |
| 6 | **Success criteria** | “Every factual claim has a valid evidence ID; no unknown ID is allowed.” | Enables automatic validation |

**Trainer message:** a longer prompt is not automatically better. A better prompt removes ambiguity and makes the result verifiable.

In [28]:
LADDER_CONTEXT = """
INCIDENT INC-1042 — STORE-104 clients cannot resolve inventory-api.internal.
E1: DNS success fell from 99.9% to 21%.
E2: WAN latency and loss remain within baseline.
E3: ACL change CHG-77 completed five minutes before the alert.
""".strip()

prompt_ladder = [
    ("Rung 0 — vague", "Fix the network."),
    ("Rung 2 — role + task", "You are a read-only NetOps copilot. Analyze the incident and propose checks."),
    ("Rung 4 — add context + constraints", (
        "You are a read-only NetOps copilot. Analyze the incident below. Use only supplied evidence; "
        "do not execute changes; separate facts, hypotheses and unknowns.\n\n" + LADDER_CONTEXT
    )),
    ("Rung 6 — add output + success criteria", (
        "You are a read-only NetOps copilot. Analyze the incident below. Return symptoms, ranked hypotheses, "
        "read-only tests, confidence, missing evidence and escalation criteria. Every material claim must cite "
        "E1, E2 or E3. A preceding change is correlation until validated.\n\n" + LADDER_CONTEXT
    )),
]

ladder_results = []
for label, prompt in prompt_ladder:
    result = run_text_demo(
        label,
        instructions="Help a NOC engineer investigate safely and concisely.",
        user_input=prompt,
        max_output_tokens=400,
    )
    if result:
        ladder_results.append({"rung": label, "output": result["text"]})

if ladder_results:
    display(pd.DataFrame(ladder_results))



--- Rung 0 — vague | gpt-5.6-luna | 4.945s ---
I can help, but I need the failure scope before making changes. Please provide:

- **What is broken?** Internet, one application, site-to-site VPN, DNS, routing, Wi‑Fi, etc.
- **Impact:** affected users/sites/devices and when it started
- **Recent changes:** config, maintenance, ISP/cloud events
- **Topology/device details:** vendor/model and management access
- **Any alerts or error messages**

Safe initial checks—read-only only:

```bash
ping <default-gateway>
ping 1.1.1.1
nslookup example.com
traceroute 1.1.1.1       # tracert on Windows
```

On network devices, collect:

```text
show interface status
show interfaces counters errors
show ip route
show ip ospf neighbor       # if applicable
show logging | tail
```

Do not reboot devices, clear counters, or alter routes until we identify the failing layer and confirm a rollback path.

--- Rung 2 — role + task | gpt-5.6-luna | 8.722s ---
Please provide the incident details, ideally:

- Sy

,rung,output
0,Rung 0 — vague,"I can help, but I need the failure scope before making changes. Please provide:\n\n- **What is broken?** Internet, one application, site-to-site VPN, DNS, routing, Wi‑Fi, etc.\n- **Impact:** affected users/sites/devices and when it started\n- **Recent changes:** config, maintenance, ISP/cloud events\n- **Topology/device details:** vendor/model and management access\n- **Any alerts or error messages**\n\nSafe initial checks—read-only only:\n\n```bash\nping <default-gateway>\nping 1.1.1.1\nnslookup example.com\ntraceroute 1.1.1.1 # tracert on Windows\n```\n\nOn network devices, collect:\n\n```text\nshow interface status\nshow interfaces counters errors\nshow ip route\nshow ip ospf neighbor # if applicable\nshow logging | tail\n```\n\nDo not reboot devices, clear counters, or alter routes until we identify the failing layer and confirm a rollback path."
1,Rung 2 — role + task,"Please provide the incident details, ideally:\n\n- Symptoms and impact: affected sites, services, users, and start time\n- Scope: devices, interfaces, VLANs/VRFs, circuits, or regions involved\n- Recent changes or maintenance\n- Alerts, error messages, and representative timestamps\n- Relevant topology or device/vendor information\n- Any checks already performed\n\nSafe read-only investigation sequence:\n\n1. **Confirm scope and timing**\n - Compare affected versus unaffected hosts/sites.\n - Correlate monitoring alerts, ticket times, and change windows.\n - Check whether the issue is isolated to management, control, or data plane.\n\n2. **Check device health**\n - Reachability and management-plane latency.\n - CPU, memory, temperature, power, and reload history.\n - HA/stack/cluster status and recent failovers.\n\n3. **Check interfaces and physical links**\n - Operational/admin state, speed/duplex, optics/transceiver diagnostics.\n - Input/output errors, CRCs, discards, flaps, carrier transitions.\n - Compare counters over time rather than relying only on totals.\n\n4. **Check Layer 2**\n - VLAN presence and trunk status.\n - STP state, topology changes, blocked ports, MAC flapping.\n - LACP member consistency and port-channel health.\n - Broadcast, multicast, or unknown-unicast spikes.\n\n5. **Check Layer 3**\n - Interface/VRF state and addressing.\n - ARP/ND resolution and duplicate-IP indicators.\n - Routing-neighbor state, route counts, withdrawals, and recent changes.\n - Presence of the expected route in the relevant RIB/FIB.\n\n6. **Check services and policy**\n - DNS/DHCP/NTP reachability where relevant.\n - ACL/firewall counters and"
2,Rung 4 — add context + constraints,"## Facts\n- DNS success for **STORE-104 clients resolving `inventory-api.internal`** dropped from **99.9% to 21%**.\n- WAN latency and packet loss remain **within baseline**.\n- ACL change **CHG-77** completed **five minutes before** the alert.\n\n## Hypotheses\n- **CHG-77 may be blocking or mispermitting DNS traffic** from STORE-104, such as UDP/TCP 53, or blocking access to the relevant DNS resolver.\n- The ACL may instead be affecting traffic to `inventory-api.internal` directly, causing the reported resolution workflow to fail or appear unsuccessful.\n- The timing supports a possible relationship between CHG-77 and the incident, but does not establish causation.\n\n## Unknowns\n- Which DNS resolver(s) STORE-104 clients use.\n- Whether failures affect UDP, TCP, or both DNS transports.\n- The source/destination interfaces, subnets, ports, and rules modified by CHG-77.\n- Whether DNS queries time out, receive `NXDOMAIN`, or receive another response.\n- Whether other stores or only STORE-104 are affected.\n- Whether the ACL has deny-hit counters or logs corresponding to the failures.\n\n## Safe read-only checks\n1. Review CHG-77’s before/after ACL diff and deployment scope.\n2. Check ACL deny logs and counters for:\n - STORE-104 client subnet(s)\n - DNS resolver IP(s)\n - UDP/TCP port 53\n3. Compare DNS success for STORE-104 with another unaffected s

## 3.5 Context engineering

Prompt engineering asks, “How should I phrase the instruction?”  
Context engineering asks, “What information should the model receive, in what structure, from which trusted source, and for how long?”

For an infrastructure investigation, assemble:

1. Incident scope and time window.
2. Relevant alerts and logs—not an unfiltered dump.
3. Current and intended configuration.
4. Recent approved changes.
5. Topology/service dependencies.
6. Applicable runbook or policy sections.
7. Clear evidence IDs and source timestamps.

### Context envelope

```text
INCIDENT: INC-1042 | SITE: STORE-104 | WINDOW: 10:02–10:12 UTC
QUESTION: Why are clients failing DNS?
EVIDENCE:
  E1 [monitoring, 10:03] DNS success rate fell from 99.9% to 21%.
  E2 [WAN telemetry, 10:04] RTT and packet loss remained within baseline.
CHANGE:
  E3 [change system, 09:58] ACL update CHG-77 completed.
BOUNDARY: Read-only analysis. No claim without evidence. State unknowns.
```

This structure improves relevance, provenance and temporal reasoning.

In [30]:
RAW_CONTEXT = """dns bad at store. users angry. maybe firewall. latency 18. change 77. success 21%."""
ENGINEERED_CONTEXT = """
INCIDENT: INC-1042 | SITE: STORE-104 | WINDOW: 10:02–10:12 UTC
QUESTION: Why are clients failing DNS resolution?
E1 [monitoring, 10:03] DNS success rate fell from 99.9% to 21%.
E2 [WAN telemetry, 10:04] RTT 18 ms and packet loss remained within baseline.
E3 [change system, 09:58] ACL update CHG-77 completed; intended rule diff is not supplied.
BOUNDARY: Read-only analysis. Cite E1–E3. State unknowns. Timing alone is not causation.
""".strip()

context_results = []
for label, context in [("raw context", RAW_CONTEXT), ("engineered context", ENGINEERED_CONTEXT)]:
    result = run_text_demo(
        f"Context engineering — {label}",
        instructions="Analyze the incident without inventing evidence. State the safest next check.",
        user_input=context,
        max_output_tokens=350,
    )
    if result:
        context_results.append({"input_design": label, "output": result["text"]})
if context_results:
    display(pd.DataFrame(context_results))



--- Context engineering — raw context | gpt-5.6-luna | 4.861s ---
### Current assessment

- Users at a store are reporting problems, and DNS is suspected.
- Reported latency is **18**—the unit and baseline are unknown.
- **Change 77** may be related, but timing and scope are not provided.
- **21% success** suggests a severe failure rate if this is request success, but the metric is undefined.
- There is not enough evidence to conclude that the firewall or change 77 is the cause.

### Safest next check

From the affected store, perform a **read-only DNS comparison**:

1. Resolve the same affected hostname using the store’s configured DNS resolver.
2. Resolve it using a known-good resolver or from a nearby unaffected store.
3. Record response time, returned address, errors/timeouts, and whether results differ.
4. Check the same application request from the affected and unaffected locations.

Do not change firewall rules or roll back change 77 yet. If DNS results are normal but applicati

,input_design,output
0,raw context,"### Current assessment\n\n- Users at a store are reporting problems, and DNS is suspected.\n- Reported latency is **18**—the unit and baseline are unknown.\n- **Change 77** may be related, but timing and scope are not provided.\n- **21% success** suggests a severe failure rate if this is request success, but the metric is undefined.\n- There is not enough evidence to conclude that the firewall or change 77 is the cause.\n\n### Safest next check\n\nFrom the affected store, perform a **read-only DNS comparison**:\n\n1. Resolve the same affected hostname using the store’s configured DNS resolver.\n2. Resolve it using a known-good resolver or from a nearby unaffected store.\n3. Record response time, returned address, errors/timeouts, and whether results differ.\n4. Check the same application request from the affected and unaffected locations.\n\nDo not change firewall rules or roll back change 77 yet. If DNS results are normal but application traffic still fails, then correlate the failures with firewall logs and the timing/scope of change 77."
1,engineered context,"### Analysis\n\nClients experienced a DNS-resolution failure during 10:02–10:12 UTC: DNS success rate dropped from 99.9% to 21% (E1).\n\nWAN health does not explain the failure at a general connectivity level: RTT and packet loss remained within baseline (E2). This makes a broad WAN outage less likely, but it does not rule out DNS-specific filtering, resolver failure, or an application/path issue.\n\nAn ACL update, CHG-77, completed at 09:58 UTC, shortly before the incident (E3). It is a plausible suspect because an incorrect rule could block or otherwise affect DNS traffic. However, the intended rule diff is unavailable, and timing alone does not establish causation.\n\n### Unknowns\n\n- Whether affected clients could reach their configured DNS resolvers.\n- Whether UDP and/or TCP DNS traffic on port 53 was blocked or altered.\n- Whether the resolvers were reachable and responding.\n- Whether CHG-77 changed the effective ACL in a way that affects DNS.\n- Whether failures occurred across all clients, sites, resolvers, or only a subset.\n\n### Safest next check\n\nPerform a read-only comparison of the **effective ACL before"


## 3.6 Structured outputs and JSON responses

Free-form prose is easy to read but difficult to validate. A schema makes required fields and allowed values explicit.

In [ ]:
class CitedClaim(BaseModel):
    text: str = Field(description="A factual statement grounded in supplied evidence") # "Interface xe-0/0/3 went down for 7 seconds."
    evidence_ids: list[str] = Field(min_length=1) # ["E1", "E2"]

class Hypothesis(BaseModel):
    cause: str # "Degraded optical path"
    supporting_evidence: list[str] # ["E1", "E2"]
    contradicting_evidence: list[str] = [] # ["E4"]
    confidence: Literal["low", "medium", "high"]
    validation_step: str

# symptoms
# [
#   {
#     "text": "Users cannot resolve the inventory application.",
#     "evidence_ids": ["E1"]
#   }
# ]
class InvestigationReport(BaseModel):
    incident_id: str
    symptoms: list[CitedClaim]
    evidence_summary: list[CitedClaim]
    hypotheses: list[Hypothesis]
    missing_evidence: list[str]
    recommended_action: str
    action_class: Literal["READ_ONLY", "REQUEST_APPROVAL", "ESCALATE"]
    execution_allowed: Literal[False] = False

class NaturalLanguageIncidentExtraction(BaseModel):
    incident_id: str
    site: str
    affected_service: str
    start_time_utc: str
    user_impact: str
    observations: list[str]
    recent_change: str | None
    unknowns: list[str]
    severity: Literal["SEV1", "SEV2", "SEV3", "SEV4", "UNDETERMINED"]

print("InvestigationReport schema")
print(json.dumps(InvestigationReport.model_json_schema(), indent=2)[:1800], "\n...")


InvestigationReport schema
{
  "$defs": {
    "CitedClaim": {
      "properties": {
        "text": {
          "description": "A factual statement grounded in supplied evidence",
          "title": "Text",
          "type": "string"
        },
        "evidence_ids": {
          "items": {
            "type": "string"
          },
          "minItems": 1,
          "title": "Evidence Ids",
          "type": "array"
        }
      },
      "required": [
        "text",
        "evidence_ids"
      ],
      "title": "CitedClaim",
      "type": "object"
    },
    "Hypothesis": {
      "properties": {
        "cause": {
          "title": "Cause",
          "type": "string"
        },
        "supporting_evidence": {
          "items": {
            "type": "string"
          },
          "title": "Supporting Evidence",
          "type": "array"
        },
        "contradicting_evidence": {
          "default": [],
          "items": {
            "type": "string"
          },
        

In [32]:
NATURAL_LANGUAGE_INCIDENT = """
At about 14:12 UTC, associates at STORE-221 said handheld replenishment terminals could sign in
but item lookups intermittently failed. Monitoring opened INC-8821 after inventory-api request
success dropped to 62%. WAN round-trip time stayed near the normal 24 ms. Change CHG-991 updated
the store firewall policy at 14:05. We do not yet have firewall hit counters, resolver health or
an application trace. Treat this as an investigation, not a confirmed firewall problem.
""".strip()

structured_incident = run_structured_demo(
    "Natural language to structured incident JSON",
    NaturalLanguageIncidentExtraction,
    instructions=(
        "Extract only facts present in the user's incident narrative. Use UNDETERMINED when severity "
        "cannot be established. Put absent diagnostic facts in unknowns. Do not diagnose a root cause."
    ),
    user_input=NATURAL_LANGUAGE_INCIDENT,
)

if structured_incident:
    display(pd.DataFrame([structured_incident.model_dump()]))



--- Natural language to structured incident JSON | gpt-5.6-terra | 3.064s ---
{
  "incident_id": "INC-8821",
  "site": "STORE-221",
  "affected_service": "handheld replenishment terminals / inventory-api",
  "start_time_utc": "About 14:12 UTC",
  "user_impact": "Associates could sign in to handheld replenishment terminals, but item lookups intermittently failed.",
  "observations": [
    "Monitoring opened INC-8821 after inventory-api request success dropped to 62%.",
    "WAN round-trip time stayed near the normal 24 ms.",
    "Change CHG-991 updated the store firewall policy at 14:05 UTC.",
    "This is an investigation and not a confirmed firewall problem."
  ],
  "recent_change": "CHG-991 updated the store firewall policy at 14:05 UTC.",
  "unknowns": [
    "Firewall hit counters are not yet available.",
    "Resolver health is not yet available.",
    "An application trace is not yet available.",
    "The root cause of the intermittent item lookup failures is unknown."
  ],
  "se

,incident_id,site,affected_service,start_time_utc,user_impact,observations,recent_change,unknowns,severity
0,INC-8821,STORE-221,handheld replenishment terminals / inventory-api,About 14:12 UTC,"Associates could sign in to handheld replenishment terminals, but item lookups intermittently failed.","[Monitoring opened INC-8821 after inventory-api request success dropped to 62%., WAN round-trip time stayed near the normal 24 ms., Change CHG-991 updated the store firewall policy at 14:05 UTC., This is an investigation and not a confirmed firewall problem.]",CHG-991 updated the store firewall policy at 14:05 UTC.,"[Firewall hit counters are not yet available., Resolver health is not yet available., An application trace is not yet available., The root cause of the intermittent item lookup failures is unknown.]",UNDETERMINED


## 3.7 Troubleshooting prompt patterns

| Pattern | Question it forces | Example |
|---|---|---|
| Symptom extraction | What was directly observed? | “List symptoms only; do not infer causes.” |
| Timeline | What happened and in what order? | “Sort evidence by timestamp and identify gaps.” |
| Differential diagnosis | What plausible causes compete? | “Give three hypotheses and a discriminating test for each.” |
| Evidence matrix | What supports or contradicts each cause? | “Cite evidence IDs in both columns.” |
| Missing-information | What prevents confidence? | “List the minimum additional read-only evidence required.” |
| Change correlation | Did a change precede the symptom? | “Treat timing as correlation until impact is validated.” |
| Safe action | What is the lowest-risk next step? | “Recommend read-only checks before remediation.” |

These patterns mirror how experienced engineers think: observe first, hypothesize second, test third, act last.

In [33]:
PATTERN_EVIDENCE = """
E1 | 10:03:02 | DNS success fell to 21% at STORE-104.
E2 | 10:03:12 | Firewall policy STORE-DNS-IN was committed under CHG-77.
E3 | 10:04:00 | WAN RTT and packet loss remained within baseline.
E4 | 10:05:10 | Current diff changes client UDP/53 from permit to deny.
E5 | 10:06:20 | The new deny rule has 1,842 matches.
""".strip()

PROMPT_PATTERNS = {
    "Symptom extraction": "List only directly observed symptoms; do not infer causes.",
    "Timeline reconstruction": "Order the events and identify temporal gaps.",
    "Differential diagnosis": "Rank three competing hypotheses and give one discriminating test each.",
    "Evidence matrix": "For each hypothesis, list supporting and contradicting evidence IDs.",
    "Missing information": "List the minimum additional read-only evidence required before RCA confidence can increase.",
    "Change correlation": "Assess whether the change is merely correlated or causally supported. State what would distinguish them.",
    "Safe next action": "Recommend the lowest-risk next action. Separate read-only checks from approval-required action.",
}

pattern_outputs = []
for pattern_name, task in PROMPT_PATTERNS.items():
    result = run_text_demo(
        f"Prompt pattern — {pattern_name}",
        instructions="Use only supplied evidence, cite evidence IDs, and do not claim an action was executed.",
        user_input=f"TASK: {task}\n\nEVIDENCE:\n{PATTERN_EVIDENCE}",
        max_output_tokens=350,
    )
    if result:
        pattern_outputs.append({"pattern": pattern_name, "output": result["text"]})
if pattern_outputs:
    display(pd.DataFrame(pattern_outputs))



--- Prompt pattern — Symptom extraction | gpt-5.6-luna | 1.78s ---
- DNS success at STORE-104 fell to 21%. [E1]
- Client UDP/53 traffic changed from permitted to denied. [E4]
- The new deny rule recorded 1,842 matches. [E5]

--- Prompt pattern — Timeline reconstruction | gpt-5.6-luna | 3.525s ---
Chronological order:

1. **10:03:02 — E1:** DNS success fell to 21% at STORE-104.
2. **10:03:12 — E2:** Firewall policy `STORE-DNS-IN` was committed under `CHG-77`.
3. **10:04:00 — E3:** WAN RTT and packet loss remained within baseline.
4. **10:05:10 — E4:** The current diff changed client UDP/53 from permit to deny.
5. **10:06:20 — E5:** The new deny rule had 1,842 matches.

Temporal gaps:

- **E1 → E2:** 10 seconds
- **E2 → E3:** 48 seconds
- **E3 → E4:** 1 minute 10 seconds
- **E4 → E5:** 1 minute 10 seconds

Total span from **E1 to E5:** 3 minutes 18 seconds.

--- Prompt pattern — Differential diagnosis | gpt-5.6-luna | 4.848s ---
| Rank | Hypothesis | Supporting evidence | One discrimina

,pattern,output
0,Symptom extraction,"- DNS success at STORE-104 fell to 21%. [E1]\n- Client UDP/53 traffic changed from permitted to denied. [E4]\n- The new deny rule recorded 1,842 matches. [E5]"
1,Timeline reconstruction,"Chronological order:\n\n1. **10:03:02 — E1:** DNS success fell to 21% at STORE-104.\n2. **10:03:12 — E2:** Firewall policy `STORE-DNS-IN` was committed under `CHG-77`.\n3. **10:04:00 — E3:** WAN RTT and packet loss remained within baseline.\n4. **10:05:10 — E4:** The current diff changed client UDP/53 from permit to deny.\n5. **10:06:20 — E5:** The new deny rule had 1,842 matches.\n\nTemporal gaps:\n\n- **E1 → E2:** 10 seconds\n- **E2 → E3:** 48 seconds\n- **E3 → E4:** 1 minute 10 seconds\n- **E4 → E5:** 1 minute 10 seconds\n\nTotal span from **E1 to E5:** 3 minutes 18 seconds."
2,Differential diagnosis,"| Rank | Hypothesis | Supporting evidence | One discriminating test |\n|---|---|---|---|\n| 1 | **CHG-77 introduced a firewall block for client DNS traffic.** | DNS success dropped to 21% at STORE-104 (E1), the firewall policy was committed immediately afterward (E2), the diff changes client UDP/53 from permit to deny (E4), and the rule has 1,842 matches (E5). | **Temporarily disable or narrowly roll back the new UDP/53 deny rule for STORE-104, then measure DNS success.** A rapid recovery would strongly support this hypothesis. |\n| 2 | **An upstream DNS resolver or DNS service is failing.** | The DNS failure itself is consistent with resolver unavailability (E1), but the evidence also shows a direct local firewall change affecting DNS (E4–E5). | **Query the configured DNS resolver from a host/path not subject to STORE-DNS-IN and compare response success with STORE-104.** If both paths fail similarly, an upstream resolver issue becomes more likely. |\n| 3 | **A broader WAN or network-connectivity problem is causing DNS failures.** | DNS success degraded (E1), but WAN RTT and packet loss remained within baseline (E3), making this less likely. | **"
3,Evidence matrix,"No hypotheses were provided, so I cannot classify evidence as supporting or contradicting each hypothesis. Please provide the hypotheses to evaluate.\n\nRelevant evidence available:\n\n- **E1:** DNS success fell to 21% at STORE-104.\n- **E2:** Firewall policy STORE-DNS-IN was committed under CHG-77.\n- **E3:** WAN RTT and packet loss remained within baseline.\n- **E4:** Client UDP/53 changed from permit to deny.\n- **E5:** The new deny rule has 1,842 matches."
4,Missing information,"Minimum additional **read-only** evidence required:\n\n1. **Firewall rule-hit and deny-log records from 10:03:02 onward**, including source clients, interfaces/zones, destination DNS servers, and timestamps, to confirm the 1,842 matches correspond to failed STORE-104 DNS traffic—not unrelated UDP/53 traffic. (E1, E4, E5)\n\n2. **DNS resolver/server query and response logs for STORE-104 over the same interval**, showing whether queries arrived and whether failures were timeouts, refusals, or server-side errors. This distinguishes a firewall-caused failure from a DNS-service problem. (E1, E3)\n\n3. **A read-only change/audit record for CHG-77**, including the exact rule scope, affected source/destination objects, deployment time, and prior configuration, to establish temporal and configuration correlation with the DNS drop. (E1, E2, E4)\n\n4. **Comparison data from unaffected stores or clients using the same DNS path during the interval**, to determine whether the impact was isolated to STORE-104 and consistent with the changed policy. (E1, E3)\n\n"
5,Change correlation,"The evidence supports a **plausible causal mechanism but not a confirmed causal claim**.\n\n- DNS success fell at 10:03:02, **before** the firewall policy commit at 10:03:12 (E1, E2). This timing weakens the claim that the commit initiated the outage.\n- The configuration diff changes client UDP/53 from permit to deny (E4), which directly matches the affected DNS traffic.\n- The deny rule has 1,842 ma

## 3.8 Production controls for hallucinations

Prompt wording is only one layer. A production design uses controls before and after the model:

1. **Trusted-source boundary:** retrieve only approved monitoring, configuration and runbook data.
2. **Data minimization:** send the relevant site, service and time window.
3. **Provenance:** assign immutable evidence IDs and timestamps.
4. **Instruction boundary:** treat retrieved text as data, not as instructions.
5. **Structured output:** restrict fields and allowed values.
6. **Citation validation:** reject evidence IDs not present in the input.
7. **Claim validation:** require citations for material statements.
8. **Uncertainty policy:** allow “insufficient evidence” and require missing evidence.
9. **Tool boundary:** start with read-only tools; separate recommendation from execution.
10. **Human gate:** an engineer validates RCA and any production change.
11. **Evaluation:** regression-test known incidents and adversarial inputs.
12. **Audit:** retain model/version, prompt, retrieved evidence, output and approval decision according to policy.

No single prompt can guarantee truth. Production reliability comes from layered controls and measurable acceptance tests.

In [ ]:
# Day1 time ends here